In [1]:
# Hugging Face에서 'qwen3 0.6b' 모델을 로컬 디렉토리에 다운로드
!huggingface-cli download Qwen/Qwen3-0.6B  --local-dir ./qwen3-0.6b

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 10 files:   0% 0/10 [00:00<?, ?it/s]Downloading 'LICENSE' to 'qwen3-0.6b/.cache/huggingface/download/DhCjcNQuMpl4FL346qr3tvNUCgY=.6634c8cc3133b3848ec74b9f275acaaa1ea618ab.incomplete'

README.md: 14.0kB [00:00, 33.5MB/s]
Download complete. Moving file to qwen3-0.6b/README.md

config.json: 100% 726/726 [00:00<00:00, 5.36MB/s]
Download complete. Moving file to qwen3-0.6b/config.json


LICENSE: 11.3kB [00:00, 36.8MB/s]

merges.txt: 0.00B [00:00, ?B/s]Download complete. Moving file to qwen3-0.6b/LICENSE


generation_config.json: 100% 239/239 [00:00<00:00, 2.26MB/s]
Download complete. Moving file to qwen3-0.6b/generation_config.json


.gitattributes: 1.57kB [00:00, 9.64MB/s]
Download complete. Moving file to qwen3-0.6b/.gitattributes
merges.txt: 1.67MB [00:00, 38.5MB/s]
Download complete. Moving file to qwen3-0.6b/merges.txt

model.safetensors:   0% 0.00/1.50G [00:00<?, ?B/s]

tokenizer.json:   0% 0.00

In [2]:
# 1. pip install llmcompressor: 신경망 압축 라이브러리인 llmcompressor를 설치합니다.
# 2. 이 라이브러리는 DeepSparse나 vLLM 같은 엔진에서 모델을 더 효율적으로 돌릴 수 있도록
#    AWQ, GPTQ 같은 양자화 알고리즘이나 프루닝(Pruning, 가지치기) 기능을 제공합니다.
# 3. 방금 전 작성하신 AWQ 압축 코드를 실행하기 위해 반드시 필요한 라이브러리입니다.

!pip install llmcompressor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.1/282.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-ml-py
    Found existing installation: nvidia-ml-py 13.590.48
    Uninstalling nvidia-ml-py-13.590.48:
      Successfully uninstalled nvidia-ml-py-13.590.48
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [3]:
# 1. 필요한 라이브러리 임포트 (데이터셋 로드, 트랜스포머, 압축 도구)
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.utils import dispatch_for_generation

# 2. 사용할 모델의 경로 설정 및 로드
MODEL_ID = "/content/qwen3-0.6b"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto", device_map="auto", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# 3. 모델 양자화에 필요한 '교정 데이터셋(Calibration Dataset)' 설정
# 양자화 시 데이터 분포를 파악하여 성능 하락을 최소화하기 위해 실제 데이터가 필요합니다.
DATASET_ID = "HuggingFaceH4/ultrachat_200k"
DATASET_SPLIT = "train_sft"
NUM_CALIBRATION_SAMPLES = 64  # 분석에 사용할 샘플 수
MAX_SEQUENCE_LENGTH = 256     # 분석할 최대 문장 길이

# 4. 데이터셋 로드 및 전처리 (채팅 템플릿 적용)
ds = load_dataset(DATASET_ID, split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]")
ds = ds.shuffle(seed=42)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
        )
    }
ds = ds.map(preprocess)

# 5. 전처리된 텍스트 데이터를 토큰화(숫자로 변환)
def tokenize(sample):
    return tokenizer(
        sample["text"],
        padding=False,
        max_length=MAX_SEQUENCE_LENGTH,
        truncation=True,
        add_special_tokens=False,
    )

# 6. AWQ 양자화 알고리즘 설정
# 가중치(Weight)는 4비트로, 활성화 함수(Activation)는 16비트로 설정(W4A16)하여 압축합니다.
recipe = [
    AWQModifier(
        ignore=["lm_head"], scheme="W4A16_ASYM", targets=["Linear"], duo_scaling="both"
    ),
]

# 7. One-shot 양자화 실행 (추가 학습 없이 한 번에 압축 진행)
oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

# 8. 양자화된 모델이 정상적으로 답변을 생성하는지 테스트 출력
print("\n\n")
print("========== SAMPLE GENERATION ==============")
dispatch_for_generation(model)
input_ids = tokenizer("Hello my name is", return_tensors="pt").input_ids.to(model.device)
output = model.generate(input_ids, max_new_tokens=100)
print(tokenizer.decode(output[0]))
print("==========================================\n\n")

# 9. 최종 압축된 모델을 디스크에 저장
SAVE_DIR = "qwen3-0.6b-awq"
model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00001-of-00003-0a1804bcb6(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

data/train_sft-00002-of-00003-ee46ed25cf(…):   0%|          | 0.00/244M [00:00<?, ?B/s]

...




========== SAMPLE GENERATION ==============


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Hello my name is John and I am a 35-year-old man. I'm in my 30s, and I have a lot of time to do things. I've been working in the tech industry for a couple of years and I'm currently in my 30s. I have a degree in engineering and I'm currently in the process of getting my degree to complete my studies. I'm in the process of getting my degree and I'm also working full-time. I'm currently in my


2026-02-02T04:54:09.455795+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 196it [00:04, 40.74it/s]


('qwen3-0.6b-awq/tokenizer_config.json',
 'qwen3-0.6b-awq/special_tokens_map.json',
 'qwen3-0.6b-awq/chat_template.jinja',
 'qwen3-0.6b-awq/vocab.json',
 'qwen3-0.6b-awq/merges.txt',
 'qwen3-0.6b-awq/added_tokens.json',
 'qwen3-0.6b-awq/tokenizer.json')

In [4]:
# 1. 모델의 전체적인 아키텍처(구조)를 콘솔에 출력합니다.
# 2. 특히 AWQ 압축을 진행한 후에는 기존의 'Linear' 레이어들이
#    양자화된 레이어 타입(예: AWQLinear 또는 W4A16 레이어)으로 교체되었는지 확인할 수 있습니다.
# 3. 모델의 각 층이 4비트로 잘 설정되었는지, lm_head처럼 제외한 부분은 그대로인지 체크하는 용도입니다.

print(model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [5]:
params = model.state_dict()

for name, param in params.items():
    # print(name)
    if 'model.layers.0.self_attn.q_proj.weight_packed' in name:
        print(name, param)

model.layers.0.self_attn.q_proj.weight_packed tensor([[ -663042696, -2041001882, -1466271644,  ..., -1213798171,
          2140433257,  2007533450],
        [ -426198890, -1786197402, -1451791724,  ...,   305686143,
          1230266945,  1662481222],
        [-1769428837, -1789027610,  -191469216,  ..., -1732711523,
          -867536931, -1229220146],
        ...,
        [-1952859523,  2007287912, -1170827123,  ..., -1144689993,
         -1802069612,  -926395773],
        [ 1283042886, -1735834980,  2077603799,  ...,  1717741414,
          -981014461, -1472821142],
        [ 1384732576,  2036763478,  2090362502,  ...,  -892757334,
          1735878551,  -123710423]], device='cuda:0', dtype=torch.int32)


In [ ]:
# 1. pip install lm_eval: 언어 모델 평가 프레임워크인 'lm-evaluation-harness'를 설치합니다.
# 2. 이 도구는 MMLU, GSM8K 등 수백 가지의 테스트를 통해 모델의 지식과 추론 능력을 측정합니다.
# 3. --user: 관리자 권한 없이 현재 사용자 계정 환경에만 라이브러리를 설치하여 환경 충돌을 방지합니다.

!pip install lm_eval --user

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 12.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=90285943fdedcc2b9fc8dd728b22d164fce179227e5c14481a099f4ea0022af5
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16862 sha256=a9df80c04abe4fbed7b5690c851a2a2ea5e6d3c22be9c15aff891f16c2df8d1a
  Stored in directory: /root/.cache/pip/wheels/7a/6f/21/

In [ ]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 498.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB ? eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 29.0 M

In [ ]:
!python3 -m lm_eval \
--model vllm \
--model_args pretrained=/content/qwen3-0.6b-awq \
--tasks mmlu \
--limit 50 \
--device cuda:0

2026-02-02:02:18:34 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-02-02:02:18:49 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
2026-02-02:02:18:55 INFO     [evaluator:211] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-02-02:02:18:55 INFO     [evaluator:236] Initializing vllm model, with arguments: {'pretrained': '/content/qwen3-0.6b-awq'}
2026-02-02 02:19:14.814842: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769998755.110091   18796 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769998755.183660   18796 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attem

In [ ]:
!python3 -m vllm.entrypoints.openai.api_server --model /content/qwen3-0.6b-awq  --port 8000

2026-02-02 02:31:25.462184: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769999485.488035   23215 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769999485.495531   23215 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769999485.513207   23215 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769999485.513238   23215 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769999485.513242   23215 computation_placer.cc:177] computation placer alr

In [ ]:
# 1. 꼬인 라이브러리 삭제 및 재설치
!pip uninstall -y protobuf google-cloud-aiplatform
!pip install -U protobuf==3.20.3
!pip install compressed-tensors transformers accelerate

Found existing installation: protobuf 6.33.5
Uninstalling protobuf-6.33.5:
  Successfully uninstalled protobuf-6.33.5
Found existing installation: google-cloud-aiplatform 1.135.0
Uninstalling google-cloud-aiplatform-1.135.0:
  Successfully uninstalled google-cloud-aiplatform-1.135.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.23.0 requires google-cloud-aiplatform[agent-engines]<2.0.0,>=1.132.0, which is not installed.
grpcio-reflection 1.76.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you 

In [ ]:
!pip install compressed-tensors transformers accelerate

In [ ]:
!pip uninstall -y compressed-tensors llmcompressor transformers
!pip install llmcompressor transformers accelerate torch

Found existing installation: compressed-tensors 0.13.0
Uninstalling compressed-tensors-0.13.0:
  Successfully uninstalled compressed-tensors-0.13.0
Found existing installation: llmcompressor 0.9.0.1
Uninstalling llmcompressor-0.9.0.1:
  Successfully uninstalled llmcompressor-0.9.0.1
Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
  Using cached llmcompressor-0.9.0.1-py3-none-any.whl.metadata (12 kB)
  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached compressed_tensors-0.13.0-py3-none-any.whl.metadata (7.0 kB)
Using cached llmcompressor-0.9.0.1-py3-none-any.whl (282 kB)
Using cached compressed_tensors-0.13.0-py3-none-any.whl (192 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier
from llmcompressor.utils import dispatch_for_generation

# Select model and load it.

QUANT_PATH = "/content/qwen3-0.6b-awq"

model = AutoModelForCausalLM.from_pretrained(QUANT_PATH, dtype="auto")

Compressing model: 196it [00:00, 274.61it/s]
Some weights of the model checkpoint at /content/qwen3-0.6b-awq were not used when initializing Qwen3ForCausalLM: ['model.layers.0.mlp.down_proj.weight_zero_point', 'model.layers.0.mlp.gate_proj.weight_zero_point', 'model.layers.0.mlp.up_proj.weight_zero_point', 'model.layers.0.self_attn.k_proj.weight_zero_point', 'model.layers.0.self_attn.o_proj.weight_zero_point', 'model.layers.0.self_attn.q_proj.weight_zero_point', 'model.layers.0.self_attn.v_proj.weight_zero_point', 'model.layers.1.mlp.down_proj.weight_zero_point', 'model.layers.1.mlp.gate_proj.weight_zero_point', 'model.layers.1.mlp.up_proj.weight_zero_point', 'model.layers.1.self_attn.k_proj.weight_zero_point', 'model.layers.1.self_attn.o_proj.weight_zero_point', 'model.layers.1.self_attn.q_proj.weight_zero_point', 'model.layers.1.self_attn.v_proj.weight_zero_point', 'model.layers.10.mlp.down_proj.weight_zero_point', 'model.layers.10.mlp.gate_proj.weight_zero_point', 'model.layers.10.m

In [ ]:
params = model.state_dict()

for name, param in params.items():
    # print(name)
    if 'model.layers.25.mlp.down_proj.weight_packed' in name:
        print(name, param)

model.layers.25.mlp.down_proj.weight_packed tensor([[ -441894202,  1772914594, -1727534964,  ...,  1821861412,
           676620808, -1202678985],
        [   17895697,   553775376,   303108369,  ..., -1254439383,
          1433970800,  1513642651],
        [-1745672619,   912533111,  1349866358,  ...,  1168536891,
           637893270,  1400059227],
        ...,
        [ 1769564826, -1635811943,  -658917546,  ...,  1177105464,
          1749142116, -1495030628],
        [ 2026674089, -1938085481, -1954973520,  ...,  1686554184,
         -2123094728,   824810630],
        [ 1484286054,  -444687031,  -900220586,  ...,  1430874711,
           896249428,  1490732405]], dtype=torch.int32)
